# Task 4 fixed-pair improvement diagnostic
Attach FS2K; run setup and dataset cells. No baseline checkpoint required. This compares two objectives on 12 fixed training pairs; results are NOT validation evidence. Training is not resumable; retain a fresh output folder.


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
from pathlib import Path
import subprocess, sys
repo = Path('/kaggle/working/GenAI_A1')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/AaizUrRehmanKazmi/GenAI_A1.git', str(repo)], check=True)
os.chdir(repo)
# For a NEW search, ensure these new files were pushed. For resume, keep the same revision.
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)
assert (repo / 'training/train_classifier.py').is_file()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-optimization.txt'], check=True)


If installation requests a restart, restart and rerun setup. For later resume, keep the same source revision; do not blindly pull changed training sources.


In [ ]:
import torch
print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'Enable a GPU in notebook settings'
assert torch.cuda.device_count() == 1, 'Restart and run setup before importing torch'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)


In [ ]:
candidates = [p.parent for p in Path('/kaggle/input').rglob('anno_train.json') if (p.parent/'photo').is_dir() and (p.parent/'sketch').is_dir()]
print(candidates)
assert len(candidates)==1, 'Attach one FS2K dataset or set raw to its exact root.'
raw = candidates[0]
subprocess.run([sys.executable,'-m','scripts.prepare_fs2k','--raw-dir',str(raw)],cwd=repo,check=True)


In [ ]:
output = Path('/kaggle/working/task4-objective-check-500')
subprocess.run([sys.executable, '-u', '-m', 'scripts.compare_task4_objectives',
    '--raw-dir', str(raw), '--output-dir', str(output), '--steps', '500',
    '--device', 'cuda'], cwd=repo, check=True)


In [ ]:
import shutil, json
from IPython.display import display, Image, FileLink
report=json.loads((output/'report.json').read_text())
for name, history in report['arms'].items(): print(name, history[-1])
display(Image(filename=str(output/'comparison.png')))
backup=shutil.make_archive('/kaggle/working/task4-objective-check_backup', 'zip', root_dir=output.parent, base_dir=output.name)
display(FileLink(backup))
